In [18]:
import pandas as pd
import numpy as np
import random

n = 1000

names = ["Rahul", "Aman", "Priya", "Sneha", "John", "Sara", "Ali", "Neha"]
cities = ["Delhi", "Mumbai", "Kolkata", "Chennai", "Bangalore", None, " ", "Hyd"]
genders = ["M", "F", "Male", "Female", None, ""]

data = []

for i in range(n):
    row = {
        "Customer_ID": f"CUST{1000 + i}" if random.random() > 0.05 else None,

        # messy names
        "Name": random.choice(names) if random.random() > 0.1 else random.choice(names).lower() + "123",

        # inconsistent gender
        "Gender": random.choice(genders),

        # age with noise
        "Age": random.choice([
            np.random.randint(18, 70),
            f"{np.random.randint(18,70)} years",
            None,
            "unknown",
            -10,  # invalid
            150   # outlier
        ]),

        # salary with symbols and missing
        "Salary": random.choice([
            round(np.random.uniform(20000, 100000), 2),
            f"₹{round(np.random.uniform(20000, 100000), 2)}",
            None,
            "NaN",
            -5000  # invalid
        ]),

        # messy dates
        "Join_Date": random.choice([
            pd.Timestamp("2020-01-01") + pd.Timedelta(days=random.randint(0,1000)),
            "12/05/2021",
            "2022-13-01",  # invalid
            None,
            "not available"
        ]),

        # city issues
        "City": random.choice(cities),

        # duplicate / noisy emails
        "Email": random.choice([
            f"user{i}@gmail.com",
            f"user{i}@gmail.com ",  # trailing space
            None,
            "invalid_email.com",
            f"user{i}@gmail"
        ]),

        # random noise column
        "Score": random.choice([
            round(np.random.uniform(0,100),2),
            None,
            "N/A",
            999,  # outlier
            -50   # invalid
        ])
    }

    data.append(row)

df = pd.DataFrame(data)

# introduce duplicates
df = pd.concat([df, df.sample(50)], ignore_index=True)

# shuffle dataset
df = df.sample(frac=1).reset_index(drop=True)

# save file
df.to_csv("messy_dataset.csv", index=False)

print("Messy dataset created: messy_dataset.csv")

Messy dataset created: messy_dataset.csv


In [206]:
df.isnull().sum()

,0
Customer_ID,0
Name,0
Gender,0
Age,0
Salary,0
Join_Date,0
City,0
Email,0
Score,0


In [37]:
df.dropna(subset='Customer_ID',inplace=True)

In [48]:
df['Gender'].unique()

array(['male', 'female', None, ''], dtype=object)

In [45]:
df['Gender']=df['Gender'].str.lower().str.strip()

In [47]:
df.replace({'m':'male','f':'female'},inplace=True)

In [102]:
df['Age'].unique()

array([64., 38., 44., 10., 29., 60., 32., 31., 63., 48., 50., 37., 67.,
       30., 25., 23., 49., 66., 54., 68., 43., 53., 19., 40., 36., 65.,
       47., 27., 57., 24., 21., 22., 35., 42., 41., 33., 20., 51., 45.,
       46., 18., 61., 59., 58., 52., 26., 56., 34., 55., 39., 28., 62.,
       69.])

In [52]:
import re
def extract_age(age):
  age_num =  re.findall('[0-9]+',str(age))
  if len(age_num)>0:
   return int(age_num[0])
  else:
    return age

In [53]:
df['Age'] = df['Age'].apply(lambda x : extract_age(x))

In [56]:
for i in ['Gender','City']:
  df[i].fillna(df[i].mode()[0],inplace=True)

/tmp/ipykernel_14007/2225060537.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[i].fillna(df[i].mode()[0],inplace=True)


In [95]:
df_age = df[df['Age'] !=  '']['Age']

In [96]:
age_median = int(df_age.dropna().astype('int64').median())

In [97]:
age_median

44

In [98]:
df.replace('',age_median,inplace=True)

In [99]:
df.loc[df['Age'] == 150, 'Age'] = age_median

In [101]:
df['Age'].fillna(age_median,inplace=True)

/tmp/ipykernel_14007/166436914.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(age_median,inplace=True)


In [139]:
df['Salary'].unique()

array([74084.,  5000., 33725., 47232., 39986., 69539., 93982., 64740.,
       99530., 50528., 90725., 67541., 24862., 56580., 57102., 67169.,
       98322., 28478., 98921., 47303., 98762., 45904., 44148., 59988.,
       30611., 64773., 30706., 41717., 85112., 34664., 23674., 35179.,
       45543., 58430., 71671., 74428., 43867., 82944., 63610., 60644.,
       60870., 73415., 61274., 63374., 44961., 32788., 68235., 98630.,
       67932., 31456., 36627., 34162., 96642., 54170., 70260., 81373.,
       51321., 24444., 53261., 84013., 20239., 30633., 87775., 76275.,
       24525., 23535., 66408., 96141., 89531., 36815., 33606., 34359.,
       43536., 24571., 94216., 90808., 22067., 26385., 43705., 74192.,
       81673., 98232., 85167., 37812., 59049., 99752., 76440., 60834.,
       48149., 40037., 28477., 78863., 22694., 35688., 63844., 96927.,
       74623., 49692., 78169., 92566., 81448., 62091., 44615., 58909.,
       79941., 66302., 45967., 56288., 98677., 62190., 28256., 82655.,
      

In [105]:
import re
def extract_sal(sal):
  sal_num =  re.findall('[0-9]+',str(sal))
  if len(sal_num)>0:
   return int(sal_num[0])
  else:
    return sal

In [106]:
df['Salary'] = df['Salary'].apply(lambda x : extract_age(x))

In [125]:
median_sal = int(df['Salary'].dropna().median())

In [126]:
median_sal

33725

In [137]:
df['Salary'].fillna(median_sal,inplace=True)

/tmp/ipykernel_14007/3775450283.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Salary'].fillna(median_sal,inplace=True)


In [136]:
print(df['Salary'].dtype)

float64


In [135]:
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

In [165]:
df['Join_Date'] = pd.to_datetime(df['Join_Date'], errors='coerce', dayfirst=True)

In [168]:
median_date = df['Join_Date'].median()

In [171]:
df['Join_Date'] = df['Join_Date'].fillna(median_date)

In [184]:
df['Join_Date'].isnull().sum()

np.int64(0)

In [185]:
df['Email'].unique()

array(['user645@gmail.com', 'user815@gmail.com', 'user716@gmail.com',
       'user760@gmail.com', 'user671@gmail.com', 'user63@gmail.com',
       'user867@gmail.com', 'user570@gmail.com', 'none',
       'user199@gmail.com', 'user708@gmail.com', 'user885@gmail.com',
       'invalid_email.com', 'user201@gmail.com', 'user523@gmail.com',
       'user606@gmail.com', 'user607@gmail.com', 'user634@gmail.com',
       'user891@gmail.com', 'user358@gmail.com', 'user855@gmail.com',
       'user75@gmail.com', 'user750@gmail.com', 'user142@gmail.com',
       'user454@gmail.com', 'user350@gmail.com', 'user534@gmail.com',
       'user558@gmail.com', 'user215@gmail.com', 'user548@gmail.com',
       'user746@gmail.com', 'user925@gmail.com', 'user46@gmail.com',
       'user773@gmail.com', 'user449@gmail.com', 'user169@gmail.com',
       'user584@gmail.com', 'user112@gmail.com', 'user224@gmail.com',
       'user939@gmail.com', 'user229@gmail.com', 'user655@gmail.com',
       'user675@gmail.com', 'user76@

In [178]:
df['Email'] = df['Email'].astype(str).str.strip().str.lower()

In [180]:
df['Email'] = df['Email'].str.replace(r'@gmail$', '@gmail.com', regex=True)

In [205]:
df['Score'].unique()

array([50.635, 55.31 , 17.14 , 30.76 , 21.   , 59.39 , 47.25 ,  8.12 ,
       11.54 , 43.64 , 68.75 , 81.68 , 84.77 , 47.27 ,  3.42 , 34.97 ,
       89.48 , 60.96 , 68.04 , 49.46 , 96.92 , 55.87 , 35.08 , 88.02 ,
       72.7  , 46.96 ,  7.47 , 46.06 ,  7.48 ,  1.39 , 64.97 , 83.77 ,
       54.97 , 35.1  , 64.68 , 17.88 , 39.34 , 95.96 , 89.92 , 75.54 ,
       25.21 , 68.08 , 41.5  ,  8.04 , 26.93 , 53.83 , 21.44 , 41.95 ,
       86.79 , 39.92 , 98.19 , 55.28 , 97.68 , 52.27 , 37.33 ,  0.53 ,
       17.95 , 89.71 , 82.46 , 91.51 , 69.59 , 87.78 , 95.76 , 44.55 ,
       50.03 , 99.38 , 12.78 , 74.13 ,  8.   ,  8.57 ,  4.05 , 54.94 ,
       11.6  , 10.58 , 26.4  , 49.05 , 61.29 , 33.83 , 71.09 , 63.51 ,
        8.08 , 26.   , 68.34 , 60.99 , 76.52 , 38.39 , 26.28 , 99.65 ,
       40.12 , 64.15 , 56.77 , 95.32 , 78.98 ,  8.36 , 99.87 , 84.84 ,
       54.58 , 74.7  , 80.09 , 90.33 , 30.89 , 10.61 , 69.47 , 66.25 ,
       73.74 , 36.13 , 32.05 , 65.68 , 40.68 , 92.68 , 40.11 , 55.53 ,
      

In [194]:
df['Score'] = pd.to_numeric(df['Score'], errors='coerce')

In [196]:
df['Score'].isnull().sum()


np.int64(366)

In [199]:
Q1 = df['Score'].quantile(0.25)
Q3 = df['Score'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df['Score'] = df['Score'].clip(lower, upper)

In [200]:
df.loc[(df['Score'] < 0) | (df['Score'] > 100), 'Score'] = pd.NA

In [201]:
median_sal = df['Score'].dropna().median()

In [202]:
median_sal

50.635

In [204]:
df['Score'].fillna(median_sal,inplace=True)

/tmp/ipykernel_14007/3523887113.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Score'].fillna(median_sal,inplace=True)
